# Módulo 2 · Clase 4 (Práctica) — CNN, data augmentation y transfer learning
### Deep Learning · Laboratorio

**Objetivos.** Cada estudiante habrá:
1. Entrenado una **CNN desde cero** en CIFAR-10.
2. Aplicado **data augmentation** y medido su efecto.
3. Hecho **transfer learning** con una ResNet preentrenada (feature extraction y fine-tuning).
4. Usado un **Vision Transformer** preentrenado como extractor de características.

**Agenda (≈ 3 h):**
| Bloque | Tema | ~min |
|---|---|---|
| 0 | Setup, datos y utilidades | 20 |
| 1 | CNN desde cero | 40 |
| 2 | Data augmentation | 30 |
| — | *Descanso* | 10 |
| 3 | Transfer learning con ResNet | 45 |
| 4 | Vision Transformer preentrenado | 25 |
| 5 | Mini-reto | 20 |


In [ ]:
# Bloque 0 · Setup
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, numpy as np, matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torchvision:", torchvision.__version__)
if device == "cpu":
    print("⚠️  Sin GPU el entrenamiento será lento. Activa GPU en Colab.")

### CIFAR-10
60.000 imágenes a color de 32×32 px, en 10 clases (avión, auto, pájaro, gato, ...). Es un clásico para experimentar con CNN.

In [ ]:
# Carga de CIFAR-10
classes = ['avión','auto','pájaro','gato','ciervo','perro','rana','caballo','barco','camión']
mean, std = (0.4914,0.4822,0.4465), (0.2470,0.2435,0.2616)
tfm_base = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])

train_ds = datasets.CIFAR10("./data", train=True,  download=True, transform=tfm_base)
test_ds  = datasets.CIFAR10("./data", train=False, download=True, transform=tfm_base)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, num_workers=2)
print("train:", len(train_ds), "| test:", len(test_ds))

In [ ]:
# Visualizar algunas imágenes
def desnormalizar(x):
    return (x * torch.tensor(std).view(3,1,1) + torch.tensor(mean).view(3,1,1)).clamp(0,1)
imgs, labels = next(iter(train_loader))
fig, ax = plt.subplots(1,8, figsize=(14,2))
for i in range(8):
    ax[i].imshow(desnormalizar(imgs[i]).permute(1,2,0)); ax[i].set_title(classes[labels[i]]); ax[i].axis('off')
plt.show()

In [ ]:
# Utilidades de entrenamiento/evaluación (reutilizables en todo el lab)
def evaluate(model, loader):
    model.eval(); correct=total=0; loss_sum=0.0; lf=nn.CrossEntropyLoss()
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            logits = model(x)
            loss_sum += lf(logits,y).item()*len(y)
            correct += (logits.argmax(1)==y).sum().item(); total += len(y)
    return loss_sum/total, correct/total

def train(model, train_loader, test_loader, epochs=5, lr=1e-3, params=None):
    lf = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(params if params is not None else model.parameters(), lr=lr)
    hist = {"val_acc":[], "val_loss":[]}
    for ep in range(epochs):
        model.train()
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            opt.zero_grad(); loss = lf(model(x), y); loss.backward(); opt.step()
        vl, va = evaluate(model, test_loader)
        hist["val_loss"].append(vl); hist["val_acc"].append(va)
        print(f"época {ep+1}/{epochs} | val_loss {vl:.3f} | val_acc {va:.3f}")
    return hist

## Bloque 1 · CNN desde cero

Construyamos una CNN con el patrón conv → ReLU → pool, repetido, y un clasificador al final.

In [ ]:
class CNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),  # 32x16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2), # 64x8x8
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),# 128x4x4
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128*4*4, n_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

cnn = CNN().to(device)
print("parámetros:", sum(p.numel() for p in cnn.parameters()))
print("salida de prueba:", cnn(torch.randn(2,3,32,32).to(device)).shape)

In [ ]:
# Entrenar la CNN (~5 épocas)
hist_cnn = train(cnn, train_loader, test_loader, epochs=5, lr=1e-3)

### 🧩 Ejercicio 1
La matriz de confusión muestra qué clases se confunden entre sí. Complétala y grafícala. ¿Qué clases cuesta más distinguir?

In [ ]:
# @title Solución
from collections import defaultdict
cnn.eval(); conf = torch.zeros(10,10, dtype=torch.int)
with torch.no_grad():
    for x,y in test_loader:
        pred = cnn(x.to(device)).argmax(1).cpu()
        for t,p in zip(y, pred): conf[t,p] += 1
plt.figure(figsize=(6,5)); plt.imshow(conf, cmap='Blues')
plt.xticks(range(10), classes, rotation=90); plt.yticks(range(10), classes)
plt.xlabel("predicción"); plt.ylabel("real"); plt.title("Matriz de confusión"); plt.colorbar(); plt.show()

## Bloque 2 · Data augmentation

Aumentar artificialmente la variabilidad del train (volteos, recortes, etc.) reduce el overfitting **sin** conseguir más datos. Solo se aplica al **train**, nunca al test.

In [ ]:
# Train con augmentation; test queda igual
tfm_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(mean, std),
])
train_aug = datasets.CIFAR10("./data", train=True, download=True, transform=tfm_aug)
train_aug_loader = DataLoader(train_aug, batch_size=128, shuffle=True, num_workers=2)

# Visualizar el efecto del augmentation sobre una misma imagen
fig, ax = plt.subplots(1,6, figsize=(12,2.2))
for i in range(6):
    x,_ = train_aug[7]   # misma imagen, distintas transformaciones aleatorias
    ax[i].imshow(desnormalizar(x).permute(1,2,0)); ax[i].axis('off')
plt.suptitle("La misma imagen con data augmentation aleatorio"); plt.show()

### 🧩 Ejercicio 2
Entrena una CNN nueva con `train_aug_loader` (mismas épocas que antes) y compara la accuracy de validación con la del Bloque 1. ¿Mejoró?

In [ ]:
# @title Solución
cnn_aug = CNN().to(device)
hist_aug = train(cnn_aug, train_aug_loader, test_loader, epochs=5, lr=1e-3)

plt.figure(figsize=(6,3.5))
plt.plot(hist_cnn["val_acc"], 'o-', label="sin augmentation")
plt.plot(hist_aug["val_acc"], 's-', label="con augmentation")
plt.xlabel("época"); plt.ylabel("val accuracy"); plt.legend(); plt.grid(True); plt.title("Efecto del data augmentation"); plt.show()

## Bloque 3 · Transfer learning con ResNet

Usaremos una **ResNet-18 preentrenada en ImageNet**. Como espera imágenes ~224×224 con la normalización de ImageNet, adaptamos las transformaciones. Para que el lab sea ágil, trabajamos con un **subconjunto** de CIFAR-10 (entrenar sobre todo el set a 224px es lento en Colab gratis).

In [ ]:
# Datos a 224x224 con normalización de ImageNet
imagenet_mean, imagenet_std = (0.485,0.456,0.406), (0.229,0.224,0.225)
tfm_224 = transforms.Compose([transforms.Resize(224), transforms.ToTensor(),
                              transforms.Normalize(imagenet_mean, imagenet_std)])
train_224 = datasets.CIFAR10("./data", train=True,  transform=tfm_224)
test_224  = datasets.CIFAR10("./data", train=False, transform=tfm_224)

# Subconjuntos para agilizar
train_sub = Subset(train_224, range(5000))
test_sub  = Subset(test_224,  range(1000))
tl_sub = DataLoader(train_sub, batch_size=64, shuffle=True, num_workers=2)
te_sub = DataLoader(test_sub,  batch_size=128, num_workers=2)
print("subconjunto train:", len(train_sub), "| test:", len(test_sub))

### Feature extraction
Cargamos la ResNet preentrenada, **congelamos** todos sus pesos y reemplazamos solo la capa final (`fc`) por una nueva de 10 clases. Entrenamos únicamente esa capa.

In [ ]:
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in resnet.parameters():            # congelar el backbone
    p.requires_grad = False
resnet.fc = nn.Linear(resnet.fc.in_features, 10)   # nueva cabeza (entrenable por defecto)
resnet = resnet.to(device)

entrenables = [p for p in resnet.parameters() if p.requires_grad]
print("parámetros entrenables:", sum(p.numel() for p in entrenables), "(solo la cabeza)")
hist_fe = train(resnet, tl_sub, te_sub, epochs=3, lr=1e-3, params=entrenables)

Fíjate qué **alta** es la accuracy con solo 3 épocas y 5000 imágenes: el backbone preentrenado ya "sabe ver".

### 🧩 Ejercicio 3 — Fine-tuning
Ahora **descongela** todo el modelo y entrénalo con un learning rate **bajo** (p. ej. `1e-4`) por unas pocas épocas. Compara con feature extraction.

In [ ]:
# @title Solución
for p in resnet.parameters():   # descongelar todo
    p.requires_grad = True
hist_ft = train(resnet, tl_sub, te_sub, epochs=3, lr=1e-4)   # lr bajo: ajuste fino
print("\nFeature extraction (última época):", round(hist_fe['val_acc'][-1],3))
print("Fine-tuning (última época):       ", round(hist_ft['val_acc'][-1],3))

## Bloque 4 · Vision Transformer preentrenado

El mismo principio de transfer learning aplica a un **ViT**. Cargamos `vit_b_16` preentrenado, congelamos el backbone y reemplazamos su cabeza de clasificación.

> Nota: el ViT es más pesado que la ResNet-18; con GPU y el subconjunto corre en pocos minutos. Si va muy lento, reduce aún más el subconjunto o el número de épocas.

In [ ]:
vit = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
for p in vit.parameters():
    p.requires_grad = False
# En torchvision, la cabeza del ViT es vit.heads.head
vit.heads.head = nn.Linear(vit.heads.head.in_features, 10)
vit = vit.to(device)

entrenables_vit = [p for p in vit.parameters() if p.requires_grad]
print("parámetros entrenables:", sum(p.numel() for p in entrenables_vit))
hist_vit = train(vit, tl_sub, te_sub, epochs=2, lr=1e-3, params=entrenables_vit)

### 🧩 Ejercicio 4
Compara en un gráfico la accuracy de validación de los tres enfoques: CNN desde cero (Bloque 1), ResNet feature extraction y ViT feature extraction. ¿Qué observas sobre el valor de preentrenar en datasets grandes?

In [ ]:
# @title Solución
plt.figure(figsize=(6,3.5))
plt.plot(hist_cnn["val_acc"], 'o-', label="CNN desde cero")
plt.plot(hist_fe["val_acc"],  's-', label="ResNet (feature extr.)")
plt.plot(hist_vit["val_acc"], '^-', label="ViT (feature extr.)")
plt.xlabel("época"); plt.ylabel("val accuracy"); plt.legend(); plt.grid(True)
plt.title("Desde cero vs. transfer learning"); plt.show()
print("Los modelos preentrenados parten de una representación visual ya aprendida:")
print("alcanzan alta accuracy con muchísimos menos datos y épocas.")

## Bloque 5 · 🏁 Mini-reto

**Objetivo: la mejor accuracy de test en CIFAR-10 que puedas.** Elige tu estrategia:
- Mejorar la CNN desde cero (más capas/canales, augmentation, más épocas, scheduler de learning rate).
- Fine-tuning completo de la ResNet sobre **todo** el dataset a 224px (más lento, pero suele superar 95%).

Define tu pipeline, entrena y reporta `evaluate(modelo, loader)[1]`.

In [ ]:
# TODO: tu mejor solución
# Pistas:
#   - scheduler: torch.optim.lr_scheduler.CosineAnnealingLR
#   - data augmentation en el train
#   - fine-tuning de resnet18/resnet50 sobre el dataset completo
...

## Cierre
Entrenaste una CNN desde cero, viste el efecto del data augmentation y comprobaste el poder del transfer learning con ResNet y ViT.

**En el Módulo 3** pasamos a NLP: cómo representar texto, redes recurrentes, el mecanismo de atención en detalle y los LLMs.